# Base paper pipeline (timed)

In [1]:
import os, sys, json, time
from datetime import datetime

ROOT = os.getcwd()
while not os.path.exists(os.path.join(ROOT, "src", "config.py")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        raise RuntimeError("Project root (folder containing src/) not found")
    ROOT = parent
sys.path.insert(0, ROOT)
OUT = os.path.join(ROOT, "base_paper")
os.makedirs(OUT, exist_ok=True)

import numpy as np
from src import config as C
from src.gci_model import pd_given_z, target_gaussian_histogram, monte_carlo_var
from src.quantum_encoding import fit_linear_angle, register_adapted_params
from src.vqc_circuit import build_gci_circuit
from src.training import train_gaussian_loader
from src.device import NoisyDevice
from src.noise_model import ControlDrift
from src.metrics import hellinger_fidelity
from src.postprocessing import credit_statistics, classical_joint_probs

REGISTERS = (2, 3)
P0, RHO, LGD, CONF = C.P0, C.RHO, C.LGD, C.CONFIDENCE

In [2]:
from src.stages import paper_grid_loader, paper_grid_gci
RESULTS = {"pipeline": "base_paper", "registers": {}}

In [3]:
TIMINGS = []


def start(stage):
    t = {"stage": stage, "start": datetime.now(), "t0": time.perf_counter()}
    print(f"[START] {stage:45s} {t['start']:%H:%M:%S}")
    return t


def stop(t, circuit_runs=0, shots=0):
    seconds = time.perf_counter() - t["t0"]
    end = datetime.now()
    TIMINGS.append({"stage": t["stage"],
                    "start": t["start"].strftime("%H:%M:%S.%f")[:-3],
                    "end": end.strftime("%H:%M:%S.%f")[:-3],
                    "seconds": round(seconds, 3),
                    "circuit_runs": int(circuit_runs), "shots": int(shots)})
    extra = f"   circuit runs = {circuit_runs}" if circuit_runs else ""
    print(f"[STOP ] {t['stage']:45s} {end:%H:%M:%S}   time = {seconds:.2f} s{extra}")


PIPELINE_START = datetime.now()
PIPELINE_T0 = time.perf_counter()
print("Pipeline started at", PIPELINE_START.strftime("%Y-%m-%d %H:%M:%S"))

Pipeline started at 2026-10-11 11:05:34


In [4]:
t = start("Stage 1 - Classical GCI model")
targets = {n: target_gaussian_histogram(n, C.Z_MAX)[1] for n in REGISTERS}
mc_losses, mc_pdf, mc_cdf, mc_var = monte_carlo_var(P0, RHO, LGD, n_trials=2_000_000, confidence=CONF)
print(f"PD(z=0) = {pd_given_z(0.0, P0, RHO):.4f}   Monte Carlo P(loss=0) = {mc_cdf[0]:.4f}   VaR95 = ${mc_var:.0f}")
stop(t)

[START] Stage 1 - Classical GCI model                 11:05:34


PD(z=0) = 0.2471   Monte Carlo P(loss=0) = 0.7497   VaR95 = $1000
[STOP ] Stage 1 - Classical GCI model                 11:05:35   time = 0.76 s


In [5]:
t = start("Stage 2 - Quantum encoding")
alpha, beta, max_err, mean_err = fit_linear_angle(P0, RHO, C.Z_MAX)
enc = {n: register_adapted_params(n, alpha, beta, C.Z_MAX) for n in REGISTERS}
print(f"alpha = {np.rad2deg(alpha):.2f} deg   beta = {np.rad2deg(beta):.2f} deg   max fit error = {max_err:.4f}")
for n in REGISTERS:
    print(f"  {n}q: alpha~ = {np.rad2deg(enc[n][0]):.2f} deg   beta~ = {np.rad2deg(enc[n][1]):.2f} deg")
stop(t)

[START] Stage 2 - Quantum encoding                    11:05:35
alpha = -3.47 deg   beta = 30.03 deg   max fit error = 0.0065
  2q: alpha~ = -6.93 deg   beta~ = 40.42 deg
  3q: alpha~ = -2.97 deg   beta~ = 40.42 deg
[STOP ] Stage 2 - Quantum encoding                    11:05:35   time = 0.00 s


In [6]:
t = start("Stage 3 - Circuit design")
circuits = {}
for n in REGISTERS:
    a_t, b_t = enc[n]
    circuits[n], _ = build_gci_circuit(n, a_t, b_t)
    print(f"  {n}q: qubits = {circuits[n].num_qubits}   depth = {circuits[n].depth()}   gates = {dict(circuits[n].count_ops())}")
stop(t)

[START] Stage 3 - Circuit design                      11:05:35
  2q: qubits = 3   depth = 6   gates = {'ry': 3, 'measure': 3, 'cry': 2, 'cx': 1, 'barrier': 1}
  3q: qubits = 4   depth = 8   gates = {'ry': 4, 'measure': 4, 'cry': 3, 'cx': 2, 'barrier': 1}
[STOP ] Stage 3 - Circuit design                      11:05:35   time = 0.00 s


In [7]:
t = start("Stage 4 - Classical training (parameter shift + Adam)")
x_nom = {}
for n in REGISTERS:
    theta, _, final_p, final_loss = train_gaussian_loader(n, targets[n], n_iterations=150, lr=0.15, seed=0)
    a_t, b_t = enc[n]
    x_nom[n] = np.array(list(theta) + [b_t, a_t])
    print(f"  {n}q: theta* = {np.round(np.rad2deg(theta), 1)} deg   loss = {final_loss:.2e}")
stop(t)

[START] Stage 4 - Classical training (parameter shift + Adam) 11:05:35


  2q: theta* = [ 90.  195.4] deg   loss = 2.28e-08


  3q: theta* = [ 90.  55. 131.] deg   loss = 4.88e-02
[STOP ] Stage 4 - Classical training (parameter shift + Adam) 11:05:36   time = 0.65 s


In [8]:
t = start("Stage 5 - Transpilation (SABRE) + noisy chip")
dev, ldev, ref, lref = {}, {}, {}, {}
for n in REGISTERS:
    dev[n] = NoisyDevice(n, "gci")
    ldev[n] = NoisyDevice(n, "loader", initial_layout=dev[n].phys[:n])
    ref[n] = dev[n].ideal_probs(x_nom[n])
    lref[n] = ldev[n].ideal_probs(x_nom[n][:n])
    ops = dict(dev[n].tqc.count_ops())
    print(f"  {n}q: transpiled depth = {dev[n].tqc.depth()}   CZ gates = {ops.get('cz', 0)}")
stop(t)

[START] Stage 5 - Transpilation (SABRE) + noisy chip  11:05:36
  2q: transpiled depth = 40   CZ gates = 8
  3q: transpiled depth = 67   CZ gates = 14
[STOP ] Stage 5 - Transpilation (SABRE) + noisy chip  11:05:36   time = 0.12 s


In [9]:
t = start("Stage 6 - Calibration: manual grid sweep (6A)")
x_cal, runs6 = {}, 0
for n in REGISTERS:
    g1 = paper_grid_loader(n, ldev[n], lref[n], x_nom[n][:n])
    x_start = x_nom[n].copy()
    x_start[:n] = g1.x_best
    g2 = paper_grid_gci(n, dev[n], ref[n], x_start)
    x_cal[n] = np.asarray(g2.x_best)
    runs = g1.n_evals + g2.n_evals
    runs6 += runs
    RESULTS["registers"][f"{n}q"] = {"calibration_runs": runs}
    print(f"  {n}q: grid sweep used {runs} circuit runs")
stop(t, runs6, runs6 * C.SHOTS_CALIBRATION)

[START] Stage 6 - Calibration: manual grid sweep (6A) 11:05:36


  2q: grid sweep used 347 circuit runs


  3q: grid sweep used 477 circuit runs
[STOP ] Stage 6 - Calibration: manual grid sweep (6A) 11:05:42   time = 5.80 s   circuit runs = 824


In [10]:
t = start("Stage 7 - Execution (raw counts, no mitigation)")
p_exec, f_exec = {}, {}
for n in REGISTERS:
    reps = [dev[n].run(x_cal[n], C.SHOTS_EXECUTION, seed=500 + 1000 * r) for r in range(C.N_REPEATS)]
    p_exec[n] = np.mean(reps, axis=0)
    f_exec[n] = [hellinger_fidelity(p, ref[n]) for p in reps]
    print(f"  {n}q: {C.N_REPEATS} repeats x {C.SHOTS_EXECUTION} shots   F_H = {np.mean(f_exec[n]):.4f} +/- {np.std(f_exec[n]):.4f}")
runs7 = len(REGISTERS) * C.N_REPEATS
stop(t, runs7, runs7 * C.SHOTS_EXECUTION)

[START] Stage 7 - Execution (raw counts, no mitigation) 11:05:42


  2q: 20 repeats x 20000 shots   F_H = 0.9002 +/- 0.0027


  3q: 20 repeats x 20000 shots   F_H = 0.9478 +/- 0.0014
[STOP ] Stage 7 - Execution (raw counts, no mitigation) 11:05:43   time = 1.46 s   circuit runs = 40


In [11]:
t = start("Stage 8 - Classical post-processing")
stats, classical = {}, {}
for n in REGISTERS:
    stats[n] = credit_statistics(p_exec[n], n, lgd=LGD, confidence=CONF)
    classical[n] = credit_statistics(classical_joint_probs(n, P0, RHO), n, lgd=LGD, confidence=CONF)
    q, c = stats[n], classical[n]
    print(f"  {n}q: PD = {q['p_default']:.4f} (classical {c['p_default']:.4f})   "
          f"E[L] = ${q['expected_loss']:.1f} (classical ${c['expected_loss']:.1f})   "
          f"VaR95 = ${q['var']:.0f} (classical ${c['var']:.0f})")
stop(t)

[START] Stage 8 - Classical post-processing           11:05:43
  2q: PD = 0.2481 (classical 0.2504)   E[L] = $248.1 (classical $250.4)   VaR95 = $1000 (classical $1000)
  3q: PD = 0.2873 (classical 0.2500)   E[L] = $287.4 (classical $250.0)   VaR95 = $1000 (classical $1000)
[STOP ] Stage 8 - Classical post-processing           11:05:43   time = 0.00 s


In [12]:
t = start("Stage 9 - Fidelity and VaR check")
for n in REGISTERS:
    f_ideal = float(np.mean(f_exec[n]))
    f_class = hellinger_fidelity(p_exec[n], classical_joint_probs(n, P0, RHO))
    RESULTS["registers"][f"{n}q"].update({
        "fidelity_vs_ideal": f_ideal, "fidelity_vs_ideal_std": float(np.std(f_exec[n])),
        "fidelity_vs_classical": float(f_class),
        "p_default": stats[n]["p_default"], "expected_loss": stats[n]["expected_loss"],
        "var95": stats[n]["var"], "classical_var95": classical[n]["var"],
        "var_matches_classical": stats[n]["var"] == classical[n]["var"],
        "decision": "not available (no acceptance check in the base paper)"})
    print(f"  {n}q: F_H vs ideal = {f_ideal:.4f}   F_H vs classical = {f_class:.4f}   "
          f"VaR matches classical: {stats[n]['var'] == classical[n]['var']}")
stop(t)

[START] Stage 9 - Fidelity and VaR check              11:05:43
  2q: F_H vs ideal = 0.9002   F_H vs classical = 0.9006   VaR matches classical: True
  3q: F_H vs ideal = 0.9478   F_H vs classical = 0.8020   VaR matches classical: True
[STOP ] Stage 9 - Fidelity and VaR check              11:05:43   time = 0.00 s


In [13]:
t = start("Drift event - repeat the manual grid sweep")
DAY2 = ControlDrift(C.DRIFT_EPS_DAY2, C.DRIFT_DELTA_DEG_DAY2)
runs_d = 0
for n in REGISTERS:
    dev2 = NoisyDevice(n, "gci", drift=DAY2)
    ldev2 = NoisyDevice(n, "loader", drift=DAY2, initial_layout=dev2.phys[:n])
    f_before = hellinger_fidelity(dev2.run(x_cal[n], C.SHOTS_FINAL_CHECK, seed=900), ref[n])
    g1 = paper_grid_loader(n, ldev2, lref[n], x_cal[n][:n])
    x_start = x_cal[n].copy()
    x_start[:n] = g1.x_best
    g2 = paper_grid_gci(n, dev2, ref[n], x_start)
    f_after = hellinger_fidelity(dev2.run(np.asarray(g2.x_best), C.SHOTS_FINAL_CHECK, seed=901), ref[n])
    runs = g1.n_evals + g2.n_evals + 2
    runs_d += runs
    RESULTS["registers"][f"{n}q"]["drift"] = {"fidelity_before": float(f_before),
                                               "fidelity_after": float(f_after),
                                               "recovery_runs": runs, "rounds": 1}
    print(f"  {n}q: after drift F_H = {f_before:.4f}  ->  after repeating the sweep F_H = {f_after:.4f}   ({runs} runs)")
stop(t, runs_d, runs_d * C.SHOTS_CALIBRATION)

[START] Drift event - repeat the manual grid sweep    11:05:43


  2q: after drift F_H = 0.6649  ->  after repeating the sweep F_H = 0.9003   (349 runs)


  3q: after drift F_H = 0.7619  ->  after repeating the sweep F_H = 0.9571   (479 runs)
[STOP ] Drift event - repeat the manual grid sweep    11:05:49   time = 5.97 s   circuit runs = 828


In [14]:
TOTAL_SECONDS = time.perf_counter() - PIPELINE_T0
PIPELINE_END = datetime.now()
print("Pipeline started :", PIPELINE_START.strftime("%H:%M:%S"))
print("Pipeline finished:", PIPELINE_END.strftime("%H:%M:%S"))
print(f"TOTAL TIME       : {TOTAL_SECONDS:.2f} s")
print()
print(f"{'stage':48s} {'start':>12s} {'end':>12s} {'seconds':>9s} {'runs':>6s}")
for r in TIMINGS:
    print(f"{r['stage']:48s} {r['start']:>12s} {r['end']:>12s} {r['seconds']:9.2f} {r['circuit_runs']:6d}")

with open(os.path.join(OUT, "timing.json"), "w") as f:
    json.dump({"pipeline": "base_paper",
               "started": PIPELINE_START.strftime("%Y-%m-%d %H:%M:%S"),
               "finished": PIPELINE_END.strftime("%Y-%m-%d %H:%M:%S"),
               "total_seconds": round(TOTAL_SECONDS, 3), "stages": TIMINGS}, f, indent=2)
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, default=float)
print("Saved ->", os.path.join(OUT, "timing.json"))
print("Saved ->", os.path.join(OUT, "results.json"))

Pipeline started : 11:05:34
Pipeline finished: 11:05:49
TOTAL TIME       : 14.87 s

stage                                                   start          end   seconds   runs
Stage 1 - Classical GCI model                    11:05:34.985 11:05:35.743      0.76      0
Stage 2 - Quantum encoding                       11:05:35.752 11:05:35.757      0.01      0
Stage 3 - Circuit design                         11:05:35.766 11:05:35.770      0.00      0
Stage 4 - Classical training (parameter shift + Adam) 11:05:35.779 11:05:36.431      0.65      0
Stage 5 - Transpilation (SABRE) + noisy chip     11:05:36.440 11:05:36.559      0.12      0
Stage 6 - Calibration: manual grid sweep (6A)    11:05:36.568 11:05:42.370      5.80    824
Stage 7 - Execution (raw counts, no mitigation)  11:05:42.380 11:05:43.839      1.46     40
Stage 8 - Classical post-processing              11:05:43.847 11:05:43.850      0.00      0
Stage 9 - Fidelity and VaR check                 11:05:43.857 11:05:43.859      0.0